# Inferência: aplicar os modelos treinados a dados nunca vistos

Este notebook **usa os modelos já treinados** (em `outputs/models/`) para pontuar dados
de **validação** — relatórios OpenFDA que os modelos nunca viram. Ele **não treina nada**:
reaplica exatamente a mesma transformação do `00_pipeline_completo.ipynb` e o **imputador
KNN ajustado só no treino**, garantindo **zero vazamento**. É autocontido (as funções de
preparação estão definidas abaixo).

**Pré-requisitos**
- Ter rodado o `00_pipeline_completo.ipynb` ao menos uma vez (gera `outputs/models/*.joblib`).
- Ter os JSON a pontuar em uma pasta (padrão: `validation/`).
- Para o imputador da faixa etária, ter os JSON de treino em `datasets/` **ou** o
  imputador já salvo em `outputs/models/age_group_knn_imputer.joblib` (salvo na 1ª execução).

> **Custo:** pontuar dados brutos exige reingerir e transformar os JSON-alvo (a validação
> tem ~1,2 GB). Use `MAX_RECORDS` para um teste rápido, ou rode numa máquina mais potente.

## 0. Imports e funções de preparação (autocontidas)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import ijson
import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Preparação de dados (mesma lógica do 00_pipeline_completo).
TARGET = "serious"

# Atributos preditores do imputador KNN da faixa etária.
KNN_FEATURES = [
    "occurcountry", "patient.patientsex", "patient.drug.drugcharacterization",
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.medicinalproduct", "patient.drug.count",
]

# Códigos de unidade de idade do OpenFDA -> fator de conversão para anos.
AGE_UNIT_TO_YEARS = {
    "800": 10.0,        # década
    "801": 1.0,         # ano
    "802": 1 / 12,      # mês
    "803": 1 / 52,      # semana
    "804": 1 / 365,     # dia
    "805": 1 / (365 * 24),  # hora
}

# Schema fixo do Parquet bruto: tipos estáveis entre lotes e arquivos.
RAW_SCHEMA = {
    "safetyreportid": pl.Utf8,
    "serious": pl.Utf8,
    "occurcountry": pl.Utf8,
    "patient.patientsex": pl.Utf8,
    "patient.patientonsetage": pl.Float64,
    "patient.patientonsetageunit": pl.Utf8,
    "patient.drug.activesubstance.activesubstancename": pl.Utf8,
    "patient.drug.drugcharacterization": pl.Utf8,
    "patient.drug.medicinalproduct": pl.Utf8,
    "patient.drug.count": pl.Int64,
}


# --------------------------------------------------------------------------- ingestão
def normalize_text_value(value):
    """Normaliza texto para MAIÚSCULAS sem espaços nas bordas; None se vazio/ausente."""
    if value is None:
        return None
    value = str(value).strip()
    return value.upper() if value else None


def join_unique_values(values):
    """Resume várias entradas em texto único e ordenado ('a | b'); 'unknown' se vazio."""
    vals = [str(v).strip() for v in values if v is not None and str(v).strip()]
    uniq = sorted(set(vals))
    return " | ".join(uniq) if uniq else "unknown"


def _to_float(value):
    """Converte para float; None quando não numérico (mantém o schema estável)."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def flatten_report(rec):
    """Achata um relatório OpenFDA em uma linha, agregando sua lista patient.drug[].

    A agregação é feita por registro (memória O(1)), pois cada relatório já carrega seus
    próprios medicamentos.
    """
    patient = rec.get("patient") or {}
    drugs = patient.get("drug") or []
    if isinstance(drugs, dict):  # um único medicamento pode vir como objeto, não lista
        drugs = [drugs]
    active = join_unique_values(
        normalize_text_value((d.get("activesubstance") or {}).get("activesubstancename"))
        for d in drugs
    )
    medic = join_unique_values(
        normalize_text_value(d.get("medicinalproduct")) for d in drugs
    )
    drugchar = join_unique_values(
        None if d.get("drugcharacterization") is None
        else str(d.get("drugcharacterization")).strip()
        for d in drugs
    )
    sid = rec.get("safetyreportid")
    return {
        "safetyreportid": str(sid) if sid is not None else None,
        "serious": None if rec.get("serious") is None else str(rec.get("serious")),
        "occurcountry": rec.get("occurcountry"),
        "patient.patientsex": None if patient.get("patientsex") is None
        else str(patient.get("patientsex")),
        "patient.patientonsetage": _to_float(patient.get("patientonsetage")),
        "patient.patientonsetageunit": None if patient.get("patientonsetageunit") is None
        else str(patient.get("patientonsetageunit")),
        "patient.drug.activesubstance.activesubstancename": active,
        "patient.drug.drugcharacterization": drugchar,
        "patient.drug.medicinalproduct": medic,
        "patient.drug.count": len(drugs),
    }


def ingest_to_parquet(json_paths, dest_parquet, batch_size=20_000, max_records=None):
    """Lê os JSON em streaming e grava as linhas achatadas em Parquet, em lotes.

    `max_records` limita o total lido (útil para um teste rápido). Retorna o nº gravado.
    """
    dest_parquet = Path(dest_parquet)
    writer = None
    batch, total = [], 0

    def _flush(rows):
        nonlocal writer, total
        if not rows:
            return
        table = pl.DataFrame(rows, schema=RAW_SCHEMA).to_arrow()
        if writer is None:
            writer = pq.ParquetWriter(dest_parquet, table.schema)
        writer.write_table(table)
        total += len(rows)

    try:
        for path in json_paths:
            with open(path, "rb") as fh:
                for rec in ijson.items(fh, "results.item"):
                    batch.append(flatten_report(rec))
                    if len(batch) >= batch_size:
                        _flush(batch)
                        batch = []
                    if max_records is not None and total + len(batch) >= max_records:
                        break
            if max_records is not None and total + len(batch) >= max_records:
                break
        _flush(batch)
    finally:
        if writer is not None:
            writer.close()
    return total


# ----------------------------------------------------------------------- transformação
def _count_pipe(column):
    """Conta itens separados por '|' numa coluna de texto (0 quando 'unknown')."""
    col = pl.col(column)
    nonempty = col.str.split("|").list.eval(
        pl.element().str.strip_chars().str.len_chars() > 0
    ).list.sum()
    return (
        pl.when(col.str.to_lowercase() == "unknown").then(0).otherwise(nonempty)
    ).cast(pl.Int64)


def build_feature_frame(parquet_path):
    """Query Polars preguiçosa de limpeza + faixa etária + feature engineering.

    Igual ao 00_pipeline_completo, porém SEM filtrar por `serious` (para permitir inferência em dados
    sem rótulo). Numéricas saem cruas. Retorna um LazyFrame; a faixa etária ainda pode
    conter 'unknown' (preenchida depois pelo imputador KNN).
    """
    sex_map = {"1": "male", "2": "female", "0": "unknown"}

    age_years = (
        pl.col("patient.patientonsetage")
        * pl.col("patient.patientonsetageunit").replace_strict(
            AGE_UNIT_TO_YEARS, default=None
        )
    )
    age_years = (
        pl.when((age_years >= 0) & (age_years <= 120)).then(age_years).otherwise(None)
    )

    return (
        pl.scan_parquet(parquet_path)
        .filter(pl.col("occurcountry").is_not_null())  # relatório sem país é descartado
        .with_columns(
            pl.col("patient.patientsex")
            .replace_strict(sex_map, default="unknown")
            .alias("patient.patientsex"),
            age_years.alias("_age_years"),
        )
        .with_columns(
            pl.when(pl.col("_age_years").is_null()).then(pl.lit("unknown"))
            .when(pl.col("_age_years") < 2).then(pl.lit("baby_early_childhood"))
            .when(pl.col("_age_years") < 12).then(pl.lit("child"))
            .when(pl.col("_age_years") < 18).then(pl.lit("adolescent"))
            .when(pl.col("_age_years") < 30).then(pl.lit("young_adult"))
            .when(pl.col("_age_years") < 60).then(pl.lit("adult"))
            .otherwise(pl.lit("elderly"))
            .alias("patient.ageGroupCalculated")
        )
        .with_columns(
            (pl.col("patient.ageGroupCalculated") == "unknown")
            .cast(pl.Int64)
            .alias("patient.ageGroupCalculated_was_imputed"),
            _count_pipe("patient.drug.activesubstance.activesubstancename")
            .alias("feature_n_active_substances"),
            _count_pipe("patient.drug.drugcharacterization").alias("feature_n_drug_types"),
            (pl.col("patient.drug.medicinalproduct").str.to_lowercase() != "unknown")
            .cast(pl.Int64)
            .alias("feature_has_medicinal_product"),
            (pl.col("patient.drug.count") > 1).cast(pl.Int64).alias("feature_multiple_drugs"),
            (pl.col("occurcountry").str.to_uppercase() == "US")
            .cast(pl.Int64)
            .alias("feature_is_usa"),
            pl.col("serious").cast(pl.Int64, strict=False).alias("serious"),
        )
        .drop("_age_years", "patient.patientonsetage", "patient.patientonsetageunit")
    )


def materialize(parquet_path):
    """Materializa a query preguiçosa em DataFrame pandas (streaming, memória limitada)."""
    return build_feature_frame(parquet_path).collect(engine="streaming").to_pandas()


def prepare_dataframe(json_paths, parquet_path, max_records=None):
    """Atalho: ingere os JSON e devolve o DataFrame transformado (sem imputar idade)."""
    ingest_to_parquet(json_paths, parquet_path, max_records=max_records)
    return materialize(parquet_path)


# ----------------------------------------------------------- imputação KNN da faixa etária
def build_age_group_imputer():
    """Cria o imputador KNN da faixa etária (ColumnTransformer + KNeighborsClassifier).

    Categóricas via One-Hot, textos de medicamentos via CountVectorizer e a contagem
    padronizada; KNN k=5 ponderado por distância. Deve ser ajustado SÓ no treino.
    """
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["occurcountry", "patient.patientsex", "patient.drug.drugcharacterization"]),
        ("active_substance",
         CountVectorizer(max_features=100, token_pattern=r"(?u)\b[\w-]+\b"),
         "patient.drug.activesubstance.activesubstancename"),
        ("medicinal_product",
         CountVectorizer(max_features=100, token_pattern=r"(?u)\b[\w-]+\b"),
         "patient.drug.medicinalproduct"),
        ("num", StandardScaler(), ["patient.drug.count"]),
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("knn", KNeighborsClassifier(n_neighbors=5, weights="distance",
                                     algorithm="brute", metric="euclidean")),
    ])


def fit_age_group_imputer(train_df):
    """Ajusta o imputador KNN nos registros de treino com faixa etária conhecida."""
    imputer = build_age_group_imputer()
    known = train_df["patient.ageGroupCalculated"] != "unknown"
    imputer.fit(
        train_df.loc[known, KNN_FEATURES],
        train_df.loc[known, "patient.ageGroupCalculated"],
    )
    return imputer


def impute_age_groups(df, fitted_imputer):
    """Preenche as faixas 'unknown' de df com o imputador já ajustado (sem refit)."""
    df = df.copy()
    unknown = df["patient.ageGroupCalculated"] == "unknown"
    if unknown.any():
        df.loc[unknown, "patient.ageGroupCalculated"] = fitted_imputer.predict(
            df.loc[unknown, KNN_FEATURES]
        )
    return df

## 1. Configuração da inferência

In [ ]:
PROJ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUTS_DIR = PROJ / "outputs"
MODELS_DIR = OUTPUTS_DIR / "models"
DATASETS_DIR = PROJ / "datasets"

# Modelo a usar (padrão: melhor experimento, EXP-05 Random Forest · Completo).
MODEL_FILE = MODELS_DIR / "EXP-05_random_forest.joblib"
# Pasta com os JSON a pontuar (dados nunca vistos).
DATA_DIR = PROJ / "validation"
# Limite de registros para teste rápido (None = todos).
MAX_RECORDS = None
# Imputador KNN persistido (reutilizado entre execuções).
IMPUTER_PATH = MODELS_DIR / "age_group_knn_imputer.joblib"

print("Modelo :", MODEL_FILE.name, "| existe:", MODEL_FILE.exists())
print("Dados  :", DATA_DIR, "|", len(list(DATA_DIR.glob("*.json"))), "JSON")
print("MAX_RECORDS:", MAX_RECORDS)

## 2. Imputador KNN da faixa etária (ajustado só no treino)

Carrega o imputador salvo, se existir; caso contrário, ajusta-o nos dados de **treino**
(mesmo split estratificado 70/30, `random_state=42`) e o persiste. Ajustar só no treino é
o que evita vazamento para os dados de validação.

In [ ]:
if IMPUTER_PATH.exists():
    age_imputer = joblib.load(IMPUTER_PATH)
    print("Imputador KNN carregado de", IMPUTER_PATH.name)
else:
    from sklearn.model_selection import train_test_split
    print("Imputador não encontrado — ajustando no treino (datasets/)...", flush=True)
    prepared = prepare_dataframe(sorted(DATASETS_DIR.glob("*.json")),
                                 OUTPUTS_DIR / "_raw_reports_train.parquet")
    prepared = prepared[prepared[TARGET].notna()]
    train_df, _ = train_test_split(
        prepared, test_size=0.30, random_state=42, stratify=prepared[TARGET])
    age_imputer = fit_age_group_imputer(train_df)
    joblib.dump(age_imputer, IMPUTER_PATH)
    print("Imputador KNN ajustado e salvo em", IMPUTER_PATH.name)

## 3. Transformar os dados a pontuar

In [ ]:
raw_infer = OUTPUTS_DIR / "_raw_reports_inference.parquet"
df = prepare_dataframe(sorted(DATA_DIR.glob("*.json")), raw_infer, max_records=MAX_RECORDS)
df = impute_age_groups(df, age_imputer)
print("Registros transformados:", df.shape)
df.head(3)

## 4. Carregar o modelo e prever

As colunas de entrada são lidas do próprio pipeline treinado (`feature_names_in_`), então
o conjunto certo (Baseline ou Completo) é selecionado automaticamente.

In [ ]:
model = joblib.load(MODEL_FILE)
cols = list(model.named_steps["preprocess"].feature_names_in_)
X = df[cols]

pred = model.predict(X)                 # rótulo (1 = grave, 2 = não-grave)
proba = model.predict_proba(X)[:, 1]    # probabilidade contínua (classe 2)

resultado = df[["safetyreportid"]].copy()
resultado["serious_pred"] = pred
resultado["proba_classe2"] = proba
dest = OUTPUTS_DIR / "predicoes_inferencia.csv"
resultado.to_csv(dest, index=False)
print("Previsões salvas em:", dest, "| linhas:", len(resultado))
resultado.head()

## 5. Métricas (quando há gabarito)

Se os registros trazem o rótulo `serious`, calculamos as mesmas métricas do
`00_pipeline_completo` (rótulo para acurácia/precisão/recall/F1; probabilidade para ROC AUC).

In [ ]:
if not df[TARGET].notna().any():
    print("Sem coluna 'serious' nos dados — pontuação gerada sem métricas.")
else:
    mask = df[TARGET].notna()
    y = df.loc[mask, TARGET].astype(int)
    yp, ypr = pred[mask.values], proba[mask.values]
    metricas = {
        "n": int(mask.sum()),
        "accuracy": accuracy_score(y, yp),
        "balanced_accuracy": balanced_accuracy_score(y, yp),
        "precision_pos": precision_score(y, yp, pos_label=1, zero_division=0),
        "recall_pos": recall_score(y, yp, pos_label=1, zero_division=0),
        "f1_pos": f1_score(y, yp, pos_label=1, zero_division=0),
        "f1_macro": f1_score(y, yp, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y, ypr),
    }
    print(MODEL_FILE.name)
    for k, v in metricas.items():
        print(f"  {k:>18}: {v:.4f}" if isinstance(v, float) else f"  {k:>18}: {v}")

    fig, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(confusion_matrix(y, yp), display_labels=[1, 2]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Matriz de confusão — {MODEL_FILE.stem}")
    ax.set_xlabel("Previsto"); ax.set_ylabel("Real")
    plt.tight_layout(); plt.show()